# Explainability Track — Step 2 & 3
## SHAP Analysis + Model Evaluation

**Owner:** Namitha

**Files to upload at the start:**
- `xgb_model.pkl`, `rf_model.pkl` — from your previous notebook
- `X_train_selected.csv`, `X_test_selected.csv`
- `y_train.csv`, `y_test.csv`

**What this notebook produces:**
- SHAP summary (beeswarm) + bar plots for both models
- Final biomarker gene ranking (`shap_biomarker_ranking.csv`)
- Pathway cross-reference table (`pathway_annotations.csv`)
- Full model comparison: accuracy/F1/ROC-AUC/confusion matrices (`model_comparison.csv` + plots)
- These feed directly into your `/explain` backend and your final report


## 1. Setup & Imports

In [ ]:
!pip install shap gseapy mygene -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
shap.initjs()


## 2. Upload & Load Everything

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
xgb_model = joblib.load("xgb_model.pkl")
rf_model  = joblib.load("rf_model.pkl")

X_train = pd.read_csv("X_train_selected.csv")
X_test  = pd.read_csv("X_test_selected.csv")
y_train = pd.read_csv("y_train.csv").iloc[:, 0]
y_test  = pd.read_csv("y_test.csv").iloc[:, 0]

gene_names = X_train.columns.tolist()

print("X_train:", X_train.shape, " X_test:", X_test.shape)
assert list(X_train.columns) == list(X_test.columns)
print("Models and data loaded, columns aligned.")


---
# PART A — SHAP Analysis


## 3. Compute SHAP Values

`TreeExplainer` is exact and fast for tree-based models (XGBoost/RF) — no
approximation needed, unlike `KernelExplainer` which you'd need for
non-tree models (e.g. Meenakshi's CNN, if she ever needs it).

We explain the **test set** — this shows which genes drive predictions on
data the model hasn't seen, which is what you actually care about for a
biomarker claim.


In [ ]:
xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_values = xgb_explainer.shap_values(X_test)

rf_explainer = shap.TreeExplainer(rf_model)
rf_shap_values = rf_explainer.shap_values(X_test)

# For binary classification, some sklearn/xgboost versions return a list [class0, class1]
# or a 3D array (n_samples, n_features, n_classes). Normalize to "shap values for class 1".
def get_positive_class_shap(shap_values):
    if isinstance(shap_values, list):
        return shap_values[1]
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        return shap_values[:, :, 1]
    return shap_values

xgb_shap_pos = get_positive_class_shap(xgb_shap_values)
rf_shap_pos  = get_positive_class_shap(rf_shap_values)

print("XGBoost SHAP shape:", np.array(xgb_shap_pos).shape)
print("RF SHAP shape:     ", np.array(rf_shap_pos).shape)


## 4. Summary Plots (Beeswarm)

The beeswarm plot is your main deliverable figure — each dot is one patient,
color = gene expression level (red=high, blue=low), x-position = impact on
the prediction. This is what typically goes in the report/poster.


In [ ]:
plt.figure()
shap.summary_plot(xgb_shap_pos, X_test, feature_names=gene_names, show=False, max_display=20)
plt.title("XGBoost — SHAP Summary (Top 20 Genes)")
plt.tight_layout()
plt.savefig("xgb_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure()
shap.summary_plot(rf_shap_pos, X_test, feature_names=gene_names, show=False, max_display=20)
plt.title("Random Forest — SHAP Summary (Top 20 Genes)")
plt.tight_layout()
plt.savefig("rf_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Bar Plots (Mean |SHAP value|) — cleaner for a report/slide


In [ ]:
plt.figure()
shap.summary_plot(xgb_shap_pos, X_test, feature_names=gene_names, plot_type="bar", show=False, max_display=20)
plt.title("XGBoost — Mean |SHAP value| (Top 20 Genes)")
plt.tight_layout()
plt.savefig("xgb_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure()
shap.summary_plot(rf_shap_pos, X_test, feature_names=gene_names, plot_type="bar", show=False, max_display=20)
plt.title("Random Forest — Mean |SHAP value| (Top 20 Genes)")
plt.tight_layout()
plt.savefig("rf_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Biomarker Gene Ranking (final ranking — supersedes Step 1's native importance)


In [ ]:
def shap_ranking(shap_values, gene_names):
    mean_abs = np.abs(shap_values).mean(axis=0)
    return pd.DataFrame({"gene": gene_names, "mean_abs_shap": mean_abs}) \
             .sort_values("mean_abs_shap", ascending=False) \
             .reset_index(drop=True)

xgb_ranking = shap_ranking(xgb_shap_pos, gene_names)
rf_ranking  = shap_ranking(rf_shap_pos, gene_names)

print("Top 20 biomarker genes — XGBoost (SHAP):")
display(xgb_ranking.head(20))

print("\nTop 20 biomarker genes — Random Forest (SHAP):")
display(rf_ranking.head(20))

xgb_ranking.to_csv("shap_biomarker_ranking_xgb.csv", index=False)
rf_ranking.to_csv("shap_biomarker_ranking_rf.csv", index=False)


In [ ]:
# How much do the two models agree on top biomarkers? Useful credibility check —
# if XGB and RF agree on top genes, that's a stronger biological signal, not just model quirk.
top_n = 20
overlap = set(xgb_ranking.head(top_n)["gene"]) & set(rf_ranking.head(top_n)["gene"])
print(f"Overlap in top {top_n} genes between XGBoost and RF: {len(overlap)}/{top_n}")
print("Shared genes:", sorted(overlap))


## 7. Pathway Cross-Referencing

Your gene column names are **Affymetrix probe IDs** (e.g. `208190_s_at`), not
gene symbols — pathway databases need gene symbols (e.g. `TP53`). So this is
a two-step process:

1. Map probe IDs → gene symbols (via `mygene.info`, which supports querying
   by Affymetrix probe/"reporter" ID directly — needs internet, which Colab has).
2. Run pathway enrichment on the mapped gene symbols using `gseapy` against
   Enrichr libraries (KEGG, Reactome, GO Biological Process).

**Note:** multiple probes can map to the same gene, and some probes may not
map to a known gene at all — both are normal and handled below.


In [ ]:
import mygene

mg = mygene.MyGeneInfo()

# Use the top ~50 genes across both models (union) for pathway analysis
top_probes = sorted(set(xgb_ranking.head(50)["gene"]) | set(rf_ranking.head(50)["gene"]))

query_result = mg.querymany(
    top_probes,
    scopes="reporter",       # Affymetrix / Illumina probe ID scope
    fields="symbol,name,entrezgene",
    species="human"
)

probe_to_symbol = {}
for r in query_result:
    if "symbol" in r:
        probe_to_symbol[r["query"]] = r["symbol"]

print(f"Mapped {len(probe_to_symbol)} / {len(top_probes)} probes to gene symbols.")
mapped_df = pd.DataFrame([
    {"probe_id": k, "gene_symbol": v} for k, v in probe_to_symbol.items()
])
display(mapped_df.head(20))
mapped_df.to_csv("probe_to_gene_symbol_mapping.csv", index=False)


In [ ]:
import gseapy as gp

gene_symbols = mapped_df["gene_symbol"].dropna().unique().tolist()
print(f"Running pathway enrichment on {len(gene_symbols)} gene symbols...")

enrichment = gp.enrichr(
    gene_list=gene_symbols,
    gene_sets=["KEGG_2021_Human", "Reactome_2022", "GO_Biological_Process_2023"],
    organism="human",
    outdir=None
)

pathway_results = enrichment.results.sort_values("Adjusted P-value")
display(pathway_results.head(20)[["Gene_set", "Term", "Overlap", "Adjusted P-value", "Genes"]])

pathway_results.to_csv("pathway_annotations.csv", index=False)


In [ ]:
# Build the final consolidated biomarker report: probe -> gene symbol -> SHAP rank -> top pathway(s)
symbol_to_pathways = {}
for _, row in pathway_results.head(30).iterrows():
    for g in row["Genes"].split(";"):
        symbol_to_pathways.setdefault(g, []).append(row["Term"])

biomarker_report = xgb_ranking.head(50).copy()
biomarker_report = biomarker_report.merge(mapped_df, left_on="gene", right_on="probe_id", how="left")
biomarker_report["top_pathways"] = biomarker_report["gene_symbol"].map(
    lambda g: "; ".join(symbol_to_pathways.get(g, [])[:3]) if pd.notna(g) else ""
)
biomarker_report = biomarker_report[["gene", "gene_symbol", "mean_abs_shap", "top_pathways"]]
biomarker_report.columns = ["probe_id", "gene_symbol", "xgb_mean_abs_shap", "top_pathways"]

display(biomarker_report.head(20))
biomarker_report.to_csv("final_biomarker_report.csv", index=False)
print("Saved: final_biomarker_report.csv — this is your report/frontend-ready biomarker table.")


---
# PART B — Evaluation


## 8. Predictions on Test Set (both models)


In [ ]:
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

rf_preds = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]


## 9. Metrics Table (Accuracy / F1 / ROC-AUC)


In [ ]:
results = pd.DataFrame([
    {
        "model": "XGBoost",
        "accuracy": accuracy_score(y_test, xgb_preds),
        "f1_score": f1_score(y_test, xgb_preds),
        "roc_auc": roc_auc_score(y_test, xgb_proba),
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, rf_preds),
        "f1_score": f1_score(y_test, rf_preds),
        "roc_auc": roc_auc_score(y_test, rf_proba),
    },
])

display(results)
results.to_csv("model_comparison.csv", index=False)


In [ ]:
print("=== XGBoost — full classification report ===")
print(classification_report(y_test, xgb_preds))

print("=== Random Forest — full classification report ===")
print(classification_report(y_test, rf_preds))


## 10. Confusion Matrices (side by side)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay(confusion_matrix(y_test, xgb_preds), display_labels=["Negative", "Positive"]) \
    .plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("XGBoost — Confusion Matrix")

ConfusionMatrixDisplay(confusion_matrix(y_test, rf_preds), display_labels=["Negative", "Positive"]) \
    .plot(ax=axes[1], cmap="Greens", colorbar=False)
axes[1].set_title("Random Forest — Confusion Matrix")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. ROC Curves (overlaid)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for name, proba, color in [("XGBoost", xgb_proba, "tab:blue"), ("Random Forest", rf_proba, "tab:green")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", color=color)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Pick the Model to Serve

Decide which model backs your `/explain` endpoint (or serve both, with SHAP
run on whichever the frontend requests). Base the choice on:
- Higher ROC-AUC / F1 on the held-out test set
- Whether the SHAP top-gene list is more biologically interpretable /
  agrees better with known pathways
- Model file size and inference speed if deployment resources are limited


In [ ]:
better_model = results.loc[results["roc_auc"].idxmax(), "model"]
print(f"Model with higher test ROC-AUC: {better_model}")
print(results)


## 13. Files Saved — What Goes Where

| File | Used by |
|---|---|
| `xgb_shap_summary.png`, `rf_shap_summary.png` | Report / poster |
| `xgb_shap_bar.png`, `rf_shap_bar.png` | Report / frontend chart reference |
| `shap_biomarker_ranking_xgb.csv`, `_rf.csv` | `/explain` endpoint, frontend gene chart |
| `probe_to_gene_symbol_mapping.csv` | Documentation / frontend gene labels |
| `pathway_annotations.csv` | Report, biomarker interpretation section |
| `final_biomarker_report.csv` | Frontend biomarker report component (directly consumable) |
| `model_comparison.csv` | Report, team evaluation section |
| `confusion_matrices.png`, `roc_curves.png` | Report / evaluation section |

Next step: build the `/explain` backend endpoint that loads the chosen model +
`TreeExplainer`, and returns `final_biomarker_report.csv`-style data as JSON
for a given input sample.
